In [ ]:
# CELL 1 — Install & Import
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install pandas openpyxl xlsxwriter matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("=" * 65)
print("  KERATOCONUS DIAGNOSIS — BELIN ABC STAGING (CORRECTED)")
print("  Bangladesh KC Dataset | Thesis-Ready Version")
print("=" * 65)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.1 MB/s eta 0:00:00
  KERATOCONUS DIAGNOSIS — BELIN ABC STAGING (CORRECTED)
  Bangladesh KC Dataset | Thesis-Ready Version


In [ ]:
# CELL 2 — Load Excel File
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ─── CONFIGURATION ──────────────────────────────────────────────────────────
# Local run: put your Excel file name here. Colab: upload when prompted.
FILE_PATH = "/content/bd_kc_data.xlsx"   # <-- CHANGE THIS if running locally

if IN_COLAB:
    from google.colab import files
    print("\n📂  Please upload your Excel file (.xlsx):")
    uploaded = files.upload()
    FILE_PATH = list(uploaded.keys())[0]
    df_raw = pd.read_excel(FILE_PATH)
else:
    df_raw = pd.read_excel(FILE_PATH)

print(f"\n✅  File loaded: {FILE_PATH}")
print(f"   Rows    : {df_raw.shape[0]}")
print(f"   Columns : {df_raw.shape[1]}")
print(f"\n   Detected columns:")
for c in df_raw.columns.tolist():
    print(f"     • {c}")




📂  Please upload your Excel file (.xlsx):


Saving bd_kc_data.xlsx to bd_kc_data (1).xlsx

✅  File loaded: bd_kc_data (1).xlsx
   Rows    : 150
   Columns : 26

   Detected columns:
     • patients_id
     • patients_name
     • Date_of_Birth
     • Exam_Date
     • RF_Front
     • RF_back
     • F_ELE_TH
     • Db
     • Dp
     • Dt
     • Da
     • BAD_D
     • ARTmax
     • B_ELE_TH
     • Thinnest_X
     • Thinnest_Y
     • Front_K_Thinnest
     • Eye
     • k_max
     • Age
     • Unnamed: 20
     • Unnamed: 21
     • Unnamed: 22
     • Astigmatism
     • Thinest_Thickness
     • Df


In [ ]:
# CELL 3 — Direct Column Mapping (EXACT names — no fragile fuzzy search)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIX #2: Replaced fragile find_col() with exact dictionary mapping.
# Research reproducibility requires deterministic column references.
# If your Excel columns differ, edit the right-hand strings below.

EXACT_COL = {
    'id'          : 'patients_id',
    'name'        : 'patients_name',
    'dob'         : 'Date_of_Birth',
    'exam_date'   : 'Exam_Date',
    'eye'         : 'Eye',                 # Already provided in your dataset
    'kmax'        : 'k_max',
    'astig'       : 'Astigmatism',
    'thickness'   : 'Thinest_Thickness',   # NOTE: your file spells "Thinest" (1 n)
    'rf_front'    : 'RF_Front',            # mm — for ABC-A
    'rf_back'     : 'RF_back',             # mm — for ABC-B
    'f_ele_th'    : 'F_ELE_TH',            # µm — anterior elevation at thinnest
    'b_ele_th'    : 'B_ELE_TH',            # µm — posterior elevation at thinnest
    'df_'         : 'Df',
    'db_'         : 'Db',
    'dp_'         : 'Dp',
    'dt_'         : 'Dt',
    'da_'         : 'Da',
    'bad_d'       : 'BAD_D',
    'artmax'      : 'ARTmax',
    'thin_x'      : 'Thinnest_X',
    'thin_y'      : 'Thinnest_Y',
    'front_k'     : 'Front_K_Thinnest',    # D — proxy for Kmax (NOT used in ABC)
    'age'         : 'Age',
}

# Verify all critical columns exist
CRITICAL = ['bad_d', 'thickness', 'rf_front', 'rf_back', 'kmax', 'eye']
missing = [k for k in CRITICAL if EXACT_COL[k] not in df_raw.columns]
if missing:
    raise ValueError(f"CRITICAL COLUMNS MISSING: {missing}\n"
                     f"Please check your Excel spelling matches EXACT_COL mapping.")

print("\n✅  All critical columns verified (exact match).")



✅  All critical columns verified (exact match).


In [ ]:
# CELL 4 — Pre-processing: Age & Eye
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

df = df_raw.copy()

# ─── A. Eye laterality ──────────────────────────────────────────────────────
# FIX #7: User already has explicit Eye column (OD/OS). No name-parsing needed.
eye_col = EXACT_COL['eye']
df['Eye_Clean'] = df[eye_col].astype(str).str.upper().str.strip()
df['Eye_Clean'] = df['Eye_Clean'].where(df['Eye_Clean'].isin(['OD','OS']), 'Unknown')

print("\n👁️  Eye distribution:")
print(df['Eye_Clean'].value_counts().to_string())

# ─── B. Age ─────────────────────────────────────────────────────────────────
# FIX: Robust age calculation with fallback to existing Age column.

def calc_age(dob_raw, exam_raw):
    """Exact age in completed years. Handles multiple date formats."""
    try:
        dob  = pd.to_datetime(dob_raw,  dayfirst=True, errors='coerce')
        exam = pd.to_datetime(exam_raw, dayfirst=True, errors='coerce')
        if pd.isnull(dob) or pd.isnull(exam):
            return np.nan
        age = exam.year - dob.year
        if (exam.month, exam.day) < (dob.month, dob.day):
            age -= 1
        return max(int(age), 0)
    except Exception:
        return np.nan

dob_col = EXACT_COL['dob']
exam_col = EXACT_COL['exam_date']

if dob_col in df.columns and exam_col in df.columns:
    df['Age_Calc'] = df.apply(lambda r: calc_age(r[dob_col], r[exam_col]), axis=1)
else:
    df['Age_Calc'] = np.nan

# Use existing Age if calculated is missing
age_col = EXACT_COL['age']
if age_col in df.columns:
    existing_age = pd.to_numeric(df[age_col], errors='coerce')
    df['Age_Final'] = df['Age_Calc'].fillna(existing_age)
else:
    df['Age_Final'] = df['Age_Calc']

ages = df['Age_Final'].dropna()
if len(ages) > 0:
    print(f"\n📅  Age: {ages.mean():.1f} ± {ages.std():.1f} yrs (n={len(ages)})")
else:
    print("\n📅  Age not available.")



👁️  Eye distribution:
Eye_Clean
Unknown    150

📅  Age: 35.4 ± 18.9 yrs (n=149)


In [ ]:
# CELL 5 — Primary 3-Class Diagnosis (BAD-D based)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# References:
#   • Belin & Ambrósio 2013 — BAD-D composite z-score concept
#   • Salomão et al. 2018 — BAD-D ≥3.05 validated for KCN cutoff
#
# Primary cutoffs:
#   BAD_D < 1.6   → NOR
#   1.6 ≤ BAD_D < 3.05 → SUSPECT
#   BAD_D ≥ 3.05  → KCN
#
# Secondary upgrade (NOR → SUSPECT) if ≥2 supporting flags present:
#   k_max > 47.2 D  |  Thinest_Thickness < 470 µm  |  Astigmatism > 1.5 D  |  ARTmax < 425

CUTOFF_SUSPECT = 1.6
CUTOFF_KCN     = 3.05
KMAX_FLAG      = 47.2
TCT_FLAG       = 470
ASTIG_FLAG     = 1.5
ARTMAX_FLAG    = 425
ARTMAX_HARD    = 300   # ARTmax < 300 = hard KCN sign (Ambrosio et al.)

def safe_float(row, col_key):
    """Safely extract numeric value from row by EXACT_COL key."""
    col = EXACT_COL.get(col_key)
    if col and col in row.index:
        try:
            v = float(row[col])
            return v if not np.isnan(v) else np.nan
        except Exception:
            return np.nan
    return np.nan

def classify_primary(row):
    """BAD-D primary classification with secondary multi-criteria upgrade."""
    bad_d = safe_float(row, 'bad_d')
    if pd.isnull(bad_d):
        return 'UNCLASSIFIED'

    # Primary rule (Belin & Ambrósio 2013; Salomão 2018)
    if bad_d >= CUTOFF_KCN:
        return 'KCN'
    elif bad_d >= CUTOFF_SUSPECT:
        return 'SUSPECT'
    else:
        # Secondary 2-of-4 flag rule for borderline NOR eyes
        k_max     = safe_float(row, 'kmax')
        thickness = safe_float(row, 'thickness')
        astig     = safe_float(row, 'astig')
        artmax    = safe_float(row, 'artmax')
        flags = sum([
            1 if (pd.notnull(k_max)     and k_max     > KMAX_FLAG)   else 0,
            1 if (pd.notnull(thickness) and thickness < TCT_FLAG)     else 0,
            1 if (pd.notnull(astig)     and astig     > ASTIG_FLAG)   else 0,
            1 if (pd.notnull(artmax)    and artmax    < ARTMAX_FLAG)  else 0,
        ])
        return 'SUSPECT' if flags >= 2 else 'NOR'

def classify_confidence(row):
    """Confidence based on distance from BAD-D cutoffs."""
    bad_d = safe_float(row, 'bad_d')
    label = row['Diagnosis']
    if pd.isnull(bad_d):
        return 'UNKNOWN'
    if label == 'KCN':
        if bad_d >= 6.0:   return 'HIGH'
        elif bad_d >= 4.0: return 'MODERATE'
        else:              return 'LOW'
    elif label == 'SUSPECT':
        margin = min(bad_d - CUTOFF_SUSPECT, CUTOFF_KCN - bad_d)
        if margin > 0.5:   return 'MODERATE'
        else:              return 'LOW'
    else:  # NOR
        if bad_d < 0.8:    return 'HIGH'
        elif bad_d < 1.4:  return 'MODERATE'
        else:              return 'LOW'

# Apply classification
df['BAD_D_num'] = pd.to_numeric(df[EXACT_COL['bad_d']], errors='coerce')
df['Diagnosis']  = df.apply(classify_primary, axis=1)
df['Confidence'] = df.apply(classify_confidence, axis=1)

# Summary
counts = df['Diagnosis'].value_counts()
total  = len(df)
print("\n" + "=" * 55)
print("  PRIMARY 3-CLASS DIAGNOSIS RESULTS")
print("=" * 55)
for cls in ['NOR', 'SUSPECT', 'KCN', 'UNCLASSIFIED']:
    n = counts.get(cls, 0)
    pct = f"({n/total*100:.1f}%)" if total > 0 else ""
    bar = "█" * int(n/total*30) if total > 0 else ""
    print(f"  {cls:<14}: {n:>4}  {pct:<8}  {bar}")

print(f"\n  Confidence breakdown:")
print(pd.crosstab(df['Diagnosis'], df['Confidence']).to_string())




  PRIMARY 3-CLASS DIAGNOSIS RESULTS
  NOR           :   55  (36.7%)   ███████████
  SUSPECT       :   53  (35.3%)   ██████████
  KCN           :   41  (27.3%)   ████████
  UNCLASSIFIED  :    1  (0.7%)    

  Confidence breakdown:
Confidence    HIGH  LOW  MODERATE  UNKNOWN
Diagnosis                                 
KCN             30    3         8        0
NOR             20    9        26        0
SUSPECT          0   42        11        0
UNCLASSIFIED     0    0         0        1


In [ ]:
# CELL 6 — Belin ABC Severity Staging (CORRECTED)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Reference: Belin MW, Duncan JK. Cornea. 2016;35(10):1355-1359
#
# Stage A — Anterior radius at thinnest point (RF_Front, mm): lower = worse
# Stage B — Posterior radius at thinnest point (RF_back, mm): lower = worse
# Stage C — Thinnest corneal thickness (µm): lower = worse
# Overall = max(Stage_A, Stage_B, Stage_C)
#
# FIX #1: _C_THRESH corrected from 360 → 300 µm per Belin & Duncan 2016.
#   Stage 0: ≥490 µm  |  Stage 1: ≥450 µm  |  Stage 2: ≥400 µm
#   Stage 3: ≥300 µm  |  Stage 4: <300 µm
#
# FIX #3: Removed forced A0B0C0 for NOR eyes. ABC is an independent
# anatomical staging system. If a NOR eye has normal anatomy, it will
# naturally score Stage 0 from the thresholds below.

_A_THRESH = [(7.25, 0), (7.05, 1), (6.35, 2), (6.15, 3)]
_B_THRESH = [(5.90, 0), (5.70, 1), (5.15, 2), (4.95, 3)]
# CORRECTED: 300 µm (not 360) for Stage 3 cutoff
_C_THRESH = [(490, 0), (450, 1), (400, 2), (300, 3)]

def _stage(value, thresholds):
    """Map value to stage 0–4 using threshold table."""
    try:
        v = float(value)
        for lower, s in thresholds:
            if v >= lower:
                return s
        return 4
    except Exception:
        return np.nan

ABC_LABELS = {
    0: 'Normal / Pre-clinical',
    1: 'Very mild KC',
    2: 'Mild – moderate KC',
    3: 'Moderate – advanced KC',
    4: 'Severe KC'
}

# Stage each parameter using exact columns
rf_front_col = EXACT_COL['rf_front']
rf_back_col  = EXACT_COL['rf_back']
thick_col    = EXACT_COL['thickness']

df['Stage_A'] = pd.to_numeric(df[rf_front_col], errors='coerce').apply(lambda v: _stage(v, _A_THRESH))
df['Stage_B'] = pd.to_numeric(df[rf_back_col],  errors='coerce').apply(lambda v: _stage(v, _B_THRESH))
df['Stage_C'] = pd.to_numeric(df[thick_col],    errors='coerce').apply(lambda v: _stage(v, _C_THRESH))

# Overall = max(A, B, C)
def abc_overall(row):
    vals = [row['Stage_A'], row['Stage_B'], row['Stage_C']]
    valid = [v for v in vals if pd.notnull(v)]
    return int(max(valid)) if valid else np.nan

df['ABC_Overall'] = df.apply(abc_overall, axis=1)
df['ABC_Label']   = df['ABC_Overall'].map(ABC_LABELS).fillna('N/A')

# ABC notation: e.g., A1B2C1
def abc_notation(row):
    a = f"A{int(row['Stage_A'])}" if pd.notnull(row['Stage_A']) else 'A?'
    b = f"B{int(row['Stage_B'])}" if pd.notnull(row['Stage_B']) else 'B?'
    c = f"C{int(row['Stage_C'])}" if pd.notnull(row['Stage_C']) else 'C?'
    return f"{a}{b}{c}"

df['ABC_Notation'] = df.apply(abc_notation, axis=1)

# ─── Print ABC distribution (KCN eyes only) ─────────────────────────────────
kcn_df = df[df['Diagnosis'] == 'KCN']
print("\n" + "=" * 55)
print("  BELIN ABC STAGING — KCN EYES ONLY")
print("=" * 55)
if len(kcn_df) > 0:
    stage_counts = kcn_df['ABC_Overall'].value_counts().sort_index()
    for s, n in stage_counts.items():
        label = ABC_LABELS.get(int(s), '')
        print(f"  Stage {int(s)}  ({label}) : {n}")
    print(f"\n  Most common ABC notations (KCN eyes):")
    top_notations = kcn_df['ABC_Notation'].value_counts().head(8)
    for notation, count in top_notations.items():
        print(f"    {notation}  :  {count} eyes")
else:
    print("  No KCN eyes found.")


  BELIN ABC STAGING — KCN EYES ONLY
  Stage 0  (Normal / Pre-clinical) : 3
  Stage 1  (Very mild KC) : 13
  Stage 2  (Mild – moderate KC) : 15
  Stage 3  (Moderate – advanced KC) : 2
  Stage 4  (Severe KC) : 8

  Most common ABC notations (KCN eyes):
    A0B0C1  :  9 eyes
    A2B2C2  :  4 eyes
    A4B4C3  :  3 eyes
    A0B0C0  :  3 eyes
    A0B0C2  :  3 eyes
    A1B1C1  :  2 eyes
    A2B1C1  :  2 eyes
    A1B0C1  :  1 eyes


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — Clinical Flags (Supporting Parameters)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BAD_SUB_THRESH = 1.6   # z-score flag for Df, Db, Dp, Dt, Da
B_ELE_SUSPECT  = 12    # µm — posterior elevation suspect threshold
B_ELE_KCN      = 16    # µm — posterior elevation KCN threshold
F_ELE_SUSPECT  = 5     # µm — anterior elevation suspect
F_ELE_KCN      = 12    # µm — anterior elevation KCN
Y_INF_THRESH   = -0.5  # mm — inferior displacement threshold

def generate_flags(row):
    """Return semicolon-separated list of active clinical flags."""
    flags = []

    # BAD sub-indices (z-score ≥ 1.6 SD = flag)
    for sub_key in ['df_', 'db_', 'dp_', 'dt_', 'da_']:
        v = safe_float(row, sub_key)
        lbl = sub_key.replace('_','').upper()
        if pd.notnull(v) and abs(v) >= BAD_SUB_THRESH:
            flags.append(f"{lbl}={v:.2f}σ")

    # Posterior elevation at thinnest point
    b_ele = safe_float(row, 'b_ele_th')
    if pd.notnull(b_ele):
        if b_ele > B_ELE_KCN:
            flags.append(f"B_ELE={b_ele:.0f}µm(KCN)")
        elif b_ele > B_ELE_SUSPECT:
            flags.append(f"B_ELE={b_ele:.0f}µm(susp)")

    # Anterior elevation at thinnest point
    f_ele = safe_float(row, 'f_ele_th')
    if pd.notnull(f_ele):
        if f_ele > F_ELE_KCN:
            flags.append(f"F_ELE={f_ele:.0f}µm(KCN)")
        elif f_ele > F_ELE_SUSPECT:
            flags.append(f"F_ELE={f_ele:.0f}µm(susp)")

    # ARTmax
    artmax = safe_float(row, 'artmax')
    if pd.notnull(artmax):
        if artmax < ARTMAX_HARD:
            flags.append(f"ARTmax={artmax:.0f}(hard-KCN)")
        elif artmax < ARTMAX_FLAG:
            flags.append(f"ARTmax={artmax:.0f}(border)")

    # Inferior displacement (Y coordinate)
    thin_y = safe_float(row, 'thin_y')
    if pd.notnull(thin_y) and thin_y < Y_INF_THRESH:
        flags.append(f"InfDisp(Y={thin_y:.2f})")

    # K_max
    kmax = safe_float(row, 'kmax')
    if pd.notnull(kmax) and kmax > KMAX_FLAG:
        flags.append(f"Kmax={kmax:.1f}D↑")

    return '; '.join(flags) if flags else 'No flags'

def count_flags(row):
    if row['Clinical_Flags'] == 'No flags':
        return 0
    return len(row['Clinical_Flags'].split(';'))

df['Clinical_Flags'] = df.apply(generate_flags, axis=1)
df['Flag_Count']     = df.apply(count_flags, axis=1)

# Thinnest point displacement magnitude
if EXACT_COL['thin_x'] in df.columns and EXACT_COL['thin_y'] in df.columns:
    tx = pd.to_numeric(df[EXACT_COL['thin_x']], errors='coerce')
    ty = pd.to_numeric(df[EXACT_COL['thin_y']], errors='coerce')
    df['Thin_Displacement_mm'] = np.sqrt(tx**2 + ty**2).round(3)
    df['Thin_Inferior'] = (ty < Y_INF_THRESH).map({True: 'Yes', False: 'No'}).fillna('Unknown')
else:
    df['Thin_Displacement_mm'] = np.nan
    df['Thin_Inferior'] = 'Unknown'

print("\n✅  Clinical flags generated.")
print(f"   Patients with ≥1 flag : {(df['Flag_Count'] > 0).sum()}")
print(f"   Mean flags per patient: {df['Flag_Count'].mean():.1f}")


✅  Clinical flags generated.
   Patients with ≥1 flag : 131
   Mean flags per patient: 3.8


In [ ]:
# CELL 8 — Statistical Summary (Thesis Table 1)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIX #4: Corrected age summary calculation. Original code had broken
# lambda/ms() mix that crashed when Age_Final was missing.

key_params = {
    'Age (yrs)'             : 'Age_Final',
    'BAD_D'                 : EXACT_COL['bad_d'],
    'k_max (D)'             : EXACT_COL['kmax'],
    'Astigmatism (D)'       : EXACT_COL['astig'],
    'Thinest_Thickness (µm)': EXACT_COL['thickness'],
    'RF_Front (mm)'         : EXACT_COL['rf_front'],
    'RF_back (mm)'          : EXACT_COL['rf_back'],
    'ARTmax'                : EXACT_COL['artmax'],
    'F_ELE_TH (µm)'         : EXACT_COL['f_ele_th'],
    'B_ELE_TH (µm)'         : EXACT_COL['b_ele_th'],
    'Df'                    : EXACT_COL['df_'],
    'Db'                    : EXACT_COL['db_'],
    'Dp'                    : EXACT_COL['dp_'],
    'Dt'                    : EXACT_COL['dt_'],
    'Da'                    : EXACT_COL['da_'],
}

classes = ['NOR', 'SUSPECT', 'KCN']
print("\n" + "=" * 80)
print("  THESIS TABLE 1 — Mean ± SD by Diagnostic Class")
print("=" * 80)
print(f"  {'Parameter':<26} {'NOR':>20} {'SUSPECT':>20} {'KCN':>20}")
print("  " + "─" * 88)

for label, col in key_params.items():
    if col is None or col not in df.columns:
        continue
    row_str = f"  {label:<26}"
    for cls in classes:
        sub = pd.to_numeric(df[df['Diagnosis'] == cls][col], errors='coerce').dropna()
        if len(sub) > 0:
            row_str += f"  {sub.mean():>8.2f} ± {sub.std():>5.2f}  "
        else:
            row_str += f"  {'N/A':>20}  "
    print(row_str)

# Eye laterality × Diagnosis
print("\n─── Eye laterality × Diagnosis ────────────────────────────────")
print(pd.crosstab(df['Eye_Clean'], df['Diagnosis']).to_string())

# ABC stage distribution (KCN eyes)
print("\n─── ABC stage distribution (KCN eyes) ──────────────────────────")
if len(kcn_df) > 0:
    for s in range(5):
        n = (kcn_df['ABC_Overall'] == s).sum()
        if n > 0:
            print(f"  Stage {s} ({ABC_LABELS[s]}): {n}")




  THESIS TABLE 1 — Mean ± SD by Diagnostic Class
  Parameter                                   NOR              SUSPECT                  KCN
  ────────────────────────────────────────────────────────────────────────────────────────
  Age (yrs)                      41.35 ± 19.91       33.06 ± 18.29       30.39 ± 16.57  
  BAD_D                           0.87 ±  0.50        1.79 ±  0.49        9.02 ±  4.62  
  k_max (D)                      44.81 ±  1.27       46.95 ±  1.77       56.93 ±  8.71  
  Astigmatism (D)                 1.27 ±  1.10        2.25 ±  1.41        4.39 ±  2.44  
  Thinest_Thickness (µm)        529.15 ± 25.46      522.51 ± 26.42      445.80 ± 80.09  
  RF_Front (mm)                   7.89 ±  0.30        7.73 ±  0.35        7.22 ±  0.58  
  RF_back (mm)                    6.60 ±  0.23        6.42 ±  0.30        5.75 ±  0.55  
  ARTmax                        459.49 ± 68.42      372.42 ± 52.80      142.32 ± 54.74  
  F_ELE_TH (µm)                   2.20 ±  1.79        3

In [ ]:
# CELL 9 — Visualisations (6-panel figure)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CLS_COLORS = {'NOR': '#4CAF50', 'SUSPECT': '#FFC107', 'KCN': '#F44336'}
STAGE_COLORS = ['#81C784','#29B6F6','#FFA726','#EF5350','#B71C1C']

fig = plt.figure(figsize=(20, 14))
fig.suptitle(
    'Bangladesh Keratoconus Dataset — Diagnosis & Belin ABC Staging',
    fontsize=16, fontweight='bold', y=0.98
)

# Panel 1: 3-class pie
ax1 = fig.add_subplot(2, 3, 1)
pie_vals   = [counts.get(c, 0) for c in ['NOR','SUSPECT','KCN']]
pie_colors = ['#4CAF50','#FFC107','#F44336']
ax1.pie(pie_vals, labels=['NOR','SUSPECT','KCN'], colors=pie_colors,
        autopct='%1.1f%%', startangle=90, textprops={'fontsize':11},
        wedgeprops={'edgecolor':'white','linewidth':2})
ax1.set_title('3-Class Distribution\n(Primary: BAD-D)', fontweight='bold', fontsize=12)

# Panel 2: BAD-D histogram
ax2 = fig.add_subplot(2, 3, 2)
bad_d_col = EXACT_COL['bad_d']
for cls, col in CLS_COLORS.items():
    sub = df[df['Diagnosis'] == cls][bad_d_col].dropna()
    ax2.hist(sub, bins=18, alpha=0.7, color=col,
             label=f"{cls} (n={len(sub)})", edgecolor='white', linewidth=0.5)
ax2.axvline(CUTOFF_SUSPECT, color='darkorange', ls='--', lw=1.8, label=f'BAD_D={CUTOFF_SUSPECT}')
ax2.axvline(CUTOFF_KCN,    color='red',        ls='--', lw=1.8, label=f'BAD_D={CUTOFF_KCN}')
ax2.legend(fontsize=9)
ax2.set_xlabel('BAD-D Score')
ax2.set_ylabel('Count')
ax2.set_title('BAD-D Distribution\nby Diagnostic Class', fontweight='bold', fontsize=12)
ax2.grid(axis='y', alpha=0.3)

# Panel 3: ABC stage bar (KCN only)
ax3 = fig.add_subplot(2, 3, 3)
kcn_plot = df[df['Diagnosis'] == 'KCN']
if len(kcn_plot) > 0:
    stage_dist = kcn_plot['ABC_Overall'].value_counts().sort_index()
    bars = ax3.bar([f"Stage {int(s)}" for s in stage_dist.index],
                   stage_dist.values,
                   color=[STAGE_COLORS[min(int(i),4)] for i in stage_dist.index],
                   edgecolor='white', linewidth=1, zorder=2)
    for bar, val in zip(bars, stage_dist.values):
        ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
                 str(val), ha='center', fontsize=11, fontweight='bold')
ax3.set_xlabel('Belin ABC Stage')
ax3.set_ylabel('Count')
ax3.set_title('ABC Severity Staging\n(KCN eyes only)', fontweight='bold', fontsize=12)
ax3.grid(axis='y', alpha=0.3, zorder=1)

# Panel 4: BAD-D vs Thickness scatter
ax4 = fig.add_subplot(2, 3, 4)
thick_col_plot = EXACT_COL['thickness']
for cls, col, marker in [('NOR','#4CAF50','o'),('SUSPECT','#FFC107','s'),('KCN','#F44336','^')]:
    sub = df[df['Diagnosis']==cls]
    x = pd.to_numeric(sub[bad_d_col], errors='coerce')
    y = pd.to_numeric(sub[thick_col_plot], errors='coerce')
    ax4.scatter(x, y, c=col, label=cls, alpha=0.75, s=50,
                marker=marker, edgecolors='white', linewidths=0.5, zorder=3)
ax4.axvline(CUTOFF_SUSPECT, color='darkorange', ls='--', lw=1.2, alpha=0.8)
ax4.axvline(CUTOFF_KCN,    color='red',        ls='--', lw=1.2, alpha=0.8)
ax4.axhline(TCT_FLAG,      color='purple',     ls='--', lw=1.2, alpha=0.8, label=f'TCT={TCT_FLAG}µm')
ax4.legend(fontsize=9)
ax4.set_xlabel('BAD-D Score')
ax4.set_ylabel('Thinnest Thickness (µm)')
ax4.set_title('BAD-D vs Thinnest Thickness\n(diagnostic space)', fontweight='bold', fontsize=12)
ax4.grid(alpha=0.3, zorder=1)

# Panel 5: RF_front & RF_back boxplots
ax5 = fig.add_subplot(2, 3, 5)
rff = EXACT_COL['rf_front']
rfb = EXACT_COL['rf_back']
plot_data_f = [pd.to_numeric(df[df['Diagnosis']==c][rff], errors='coerce').dropna() for c in classes]
plot_data_b = [pd.to_numeric(df[df['Diagnosis']==c][rfb], errors='coerce').dropna() for c in classes]
pos_f = [1, 4, 7]; pos_b = [2, 5, 8]
bp1 = ax5.boxplot(plot_data_f, positions=pos_f, widths=0.8, patch_artist=True, showfliers=True)
bp2 = ax5.boxplot(plot_data_b, positions=pos_b, widths=0.8, patch_artist=True, showfliers=True)
for patch, col in zip(bp1['boxes'], ['#4CAF50','#FFC107','#F44336']):
    patch.set_facecolor(col); patch.set_alpha(0.7)
for patch, col in zip(bp2['boxes'], ['#4CAF50','#FFC107','#F44336']):
    patch.set_facecolor(col); patch.set_alpha(0.4)
ax5.set_xticks([1.5, 4.5, 7.5]); ax5.set_xticklabels(classes)
ax5.set_ylabel('Radius (mm)')
ax5.legend(handles=[mpatches.Patch(fc='gray',alpha=0.7,label='RF_Front'),
                    mpatches.Patch(fc='gray',alpha=0.4,label='RF_back')], fontsize=9)
ax5.set_title('RF_Front & RF_back\n(Belin A & B)', fontweight='bold', fontsize=12)
ax5.grid(axis='y', alpha=0.3)

# Panel 6: Age distribution
ax6 = fig.add_subplot(2, 3, 6)
for cls, col in CLS_COLORS.items():
    sub = df[df['Diagnosis']==cls]['Age_Final'].dropna()
    if len(sub) > 0:
        ax6.hist(sub, bins=12, alpha=0.7, color=col,
                 label=f"{cls} (μ={sub.mean():.0f}±{sub.std():.0f}yrs)",
                 edgecolor='white', linewidth=0.5)
ax6.set_xlabel('Age (years)'); ax6.set_ylabel('Count')
ax6.legend(fontsize=9)
ax6.set_title('Age Distribution\nby Diagnostic Class', fontweight='bold', fontsize=12)
ax6.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('kc_diagnosis_charts_CORRECTED.png', dpi=150, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print("✅  Charts saved: kc_diagnosis_charts_CORRECTED.png")



✅  Charts saved: kc_diagnosis_charts_CORRECTED.png


In [ ]:
# CELL 10 — Export to Excel (7 Sheets)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FIX #5: Simplified column selection for Clinical Flags sheet to prevent
# list-comprehension crash when optional columns are absent.

OUT_FILE = 'bd_kc_diagnosed_ABC_CORRECTED.xlsx'

# Build output column order
output_cols = []
for k in ['id','name','eye']:
    c = EXACT_COL.get(k)
    if c and c in df.columns: output_cols.append(c)

output_cols += ['Eye_Clean', 'Age_Final']

for k in ['dob','exam_date','kmax','astig','thickness','rf_front','rf_back',
          'f_ele_th','b_ele_th','df_','db_','dp_','dt_','da_','bad_d',
          'artmax','thin_x','thin_y','front_k']:
    c = EXACT_COL.get(k)
    if c and c in df.columns: output_cols.append(c)

output_cols += ['Thin_Displacement_mm', 'Thin_Inferior',
                'Diagnosis', 'Confidence', 'Clinical_Flags', 'Flag_Count',
                'Stage_A', 'Stage_B', 'Stage_C',
                'ABC_Overall', 'ABC_Notation', 'ABC_Label']

# Deduplicate while preserving order
seen = set()
output_cols = [c for c in output_cols if not (c in seen or seen.add(c))]

# Ensure all exist in df
output_cols = [c for c in output_cols if c in df.columns]

# Color fills for diagnosis
FILL = {
    'NOR'          : 'C6EFCE',
    'SUSPECT'      : 'FFEB9C',
    'KCN'          : 'FFC7CE',
    'UNCLASSIFIED' : 'D9D9D9',
}
HEADER_COLOR = '2F5496'
HEADER_FONT  = 'FFFFFF'

def style_sheet(ws, diag_col_name):
    """Apply header styling + row colour coding."""
    # Header row
    for cell in ws[1]:
        cell.font      = Font(bold=True, color=HEADER_FONT, size=10)
        cell.fill      = PatternFill('solid', start_color=HEADER_COLOR)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[1].height = 30

    # Find diagnosis column index (1-based)
    diag_idx = None
    for i, cell in enumerate(ws[1], 1):
        if cell.value == diag_col_name:
            diag_idx = i
            break

    if diag_idx:
        for row in ws.iter_rows(min_row=2):
            diag_val = row[diag_idx - 1].value
            if diag_val in FILL:
                fill = PatternFill('solid', start_color=FILL[diag_val])
                for cell in row:
                    cell.fill = fill
                    cell.alignment = Alignment(horizontal='center', vertical='center')

    # Auto column width
    for col_cells in ws.columns:
        max_len = max((len(str(cell.value)) if cell.value is not None else 0) for cell in col_cells)
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max_len + 3, 30)

with pd.ExcelWriter(OUT_FILE, engine='openpyxl') as writer:

    # Sheet 1: All patients
    df[output_cols].to_excel(writer, sheet_name='All_Patients', index=False)

    # Sheets 2–4: per class
    for cls in ['NOR', 'SUSPECT', 'KCN']:
        sub = df[df['Diagnosis'] == cls][output_cols].reset_index(drop=True)
        if len(sub) > 0:
            sub.to_excel(writer, sheet_name=cls, index=False)

    # Sheet 5: Summary statistics
    summary_rows = []
    for cls in classes:
        sub = df[df['Diagnosis'] == cls]
        n = len(sub)
        def ms(col_name):
            if col_name and col_name in sub.columns:
                s = pd.to_numeric(sub[col_name], errors='coerce').dropna()
                return f"{s.mean():.2f} ± {s.std():.2f}" if len(s) > 0 else 'N/A'
            return 'N/A'

        summary_rows.append({
            'Class'           : cls,
            'n (eyes)'        : n,
            'Percentage'      : f"{n/total*100:.1f}%" if total > 0 else "N/A",
            'Age mean±SD'     : ms('Age_Final'),
            'BAD_D mean±SD'   : ms(EXACT_COL['bad_d']),
            'k_max mean±SD'   : ms(EXACT_COL['kmax']),
            'TCT mean±SD'     : ms(EXACT_COL['thickness']),
            'RF_Front mean±SD': ms(EXACT_COL['rf_front']),
            'RF_back mean±SD' : ms(EXACT_COL['rf_back']),
            'ARTmax mean±SD'  : ms(EXACT_COL['artmax']),
        })
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary_Statistics', index=False)

    # Sheet 6: ABC staging breakdown
    if len(kcn_df) > 0:
        abc_rows = []
        for s in range(5):
            sub_s = kcn_df[kcn_df['ABC_Overall'] == s]
            if len(sub_s) > 0:
                abc_rows.append({
                    'Stage'     : s,
                    'Meaning'   : ABC_LABELS[s],
                    'Count'     : len(sub_s),
                    'Percentage': f"{len(sub_s)/len(kcn_df)*100:.1f}%",
                    'Common_ABC': sub_s['ABC_Notation'].value_counts().index[0] if len(sub_s) > 0 else 'N/A',
                })
        if abc_rows:
            pd.DataFrame(abc_rows).to_excel(writer, sheet_name='ABC_Staging', index=False)

    # Sheet 7: Clinical Flags
    flag_cols = [c for c in output_cols if c in df.columns]
    if 'Clinical_Flags' not in flag_cols: flag_cols.append('Clinical_Flags')
    if 'Flag_Count' not in flag_cols:     flag_cols.append('Flag_Count')
    flag_cols = list(dict.fromkeys(flag_cols))  # dedup
    df[flag_cols].to_excel(writer, sheet_name='Clinical_Flags', index=False)

# Apply styling
wb = load_workbook(OUT_FILE)
for sh_name in ['All_Patients', 'NOR', 'SUSPECT', 'KCN', 'Clinical_Flags']:
    if sh_name in wb.sheetnames:
        style_sheet(wb[sh_name], 'Diagnosis')
for ws in wb.worksheets:
    ws.freeze_panes = 'A2'
wb.save(OUT_FILE)

print("\n" + "=" * 55)
print("  EXPORT COMPLETE")
print("=" * 55)
print(f"  File     : {OUT_FILE}")
print(f"  Sheets   : 7")
print(f"    1. All_Patients      — full dataset (colour coded)")
print(f"    2. NOR               — {counts.get('NOR',0)} eyes (green)")
print(f"    3. SUSPECT           — {counts.get('SUSPECT',0)} eyes (yellow)")
print(f"    4. KCN               — {counts.get('KCN',0)} eyes (red)")
print(f"    5. Summary_Statistics — mean ± SD table (thesis Table 1)")
print(f"    6. ABC_Staging       — Belin ABC stage distribution")
print(f"    7. Clinical_Flags    — per-patient flag report")

if IN_COLAB:
    files.download(OUT_FILE)
    files.download('kc_diagnosis_charts_CORRECTED.png')
    print("\n🎉  Both files downloaded.")
else:
    print(f"\n📁  Files saved in current folder:")
    print(f"     • {OUT_FILE}")
    print(f"     • kc_diagnosis_charts_CORRECTED.png")
    print("    (Upload these to your thesis folder)")




  EXPORT COMPLETE
  File     : bd_kc_diagnosed_ABC_CORRECTED.xlsx
  Sheets   : 7
    1. All_Patients      — full dataset (colour coded)
    2. NOR               — 55 eyes (green)
    3. SUSPECT           — 53 eyes (yellow)
    4. KCN               — 41 eyes (red)
    5. Summary_Statistics — mean ± SD table (thesis Table 1)
    6. ABC_Staging       — Belin ABC stage distribution
    7. Clinical_Flags    — per-patient flag report


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉  Both files downloaded.
